## The purpose of this file is to find a better model for query generation part.

In [1]:
import numpy as np
from langchain_ollama import ChatOllama
from sklearn.model_selection import ParameterGrid
from sklearn.model_selection import GridSearchCV

from rag_flow import RAGFlow
import pandas as pd
from time import perf_counter
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt
import requests
from eval import Eval
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity


from sklearn.metrics import accuracy_score, precision_score, recall_score, confusion_matrix

/Users/vilya/Desktop/capstone_project/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-10-09 23:41:34,958 | INFO | No device provided, using mps
2026-10-09 23:41:35,089 | INFO | HTTP Request: GET https://huggingface.co/api/agent-harnesses "HTTP/1.1 200 OK"
2026-10-09 23:41:35,176 | INFO | HTTP Request: HEAD https://huggingface.co/cross-encoder/ms-marco-MiniLM-L6-v2/resolve/main/modules.json "HTTP/1.1 404 Not Found"
2026-10-09 23:41:35,176 | WARNING | Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
2026-10-09 23:41:35,178 | INFO | No modules.json found for cross-encoder/ms-marco-MiniLM-L6-v2, initializing a new CrossEncoder model.
2026-10-09 23:41:35,267 | INFO | HTTP Request: GET https://huggingface.co/ap

In [2]:
response = requests.get("http://localhost:11434/api/tags")
models = response.json()["models"]

for model in models:
    print(f"{model['name']}: {model['size'] / 1024**3:.2f} GB")

qwen3:8b: 4.87 GB
llama3.2:3b: 1.88 GB
nomic-embed-text:latest: 0.26 GB
gpt-oss:20b: 12.85 GB


For this part of the system, I will compare two models: qwen3 and gpt-oss.

In [3]:
evaluation_df = pd.DataFrame({
    "qwen": {
            "mean_query_relevancy": None,
            "retrieval_success_rate": None,
            "mean_latency": None
    },
    "gpt": {
        "mean_query_relevancy": None,
        "retrieval_success_rate": None,
        "mean_latency": None
    }
})

In [4]:
df = pd.read_csv("test_data/arxiv_classifier_300_questions.csv")
questions = df[df["label"] == "arxiv"]["question"]
questions.head(3)


0    What are the main limitations of economic grow...
2    What are the main limitations of operating sys...
3    How can researchers evaluate optical interfere...
Name: question, dtype: str

In [5]:
qwen = ChatOllama(
    model="qwen3:8b",
    base_url="http://localhost:11434"
)

gpt = ChatOllama(
    model="gpt-oss:20b",
    base_url="http://localhost:11434"
)
classifier = ChatOllama(
    model="llama3.2:3b",
    base_url="http://localhost:11434",
    temperature=0
)


In [18]:
models = {
    "qwen": qwen,
    "gpt": gpt }
eval_model = SentenceTransformer("all-MiniLM-L6-v2")
latencies = {model: [] for model in models}
relevance_scores = {model: [] for model in models}
# 1 success 0 not found (timeout)
# Please note, 0 might also indicate that the system was not able to connect to the arxiv
# web page, which can happen for multiple reasons, so 0 is not an exclusive indicator of a query that
# was not able to find a title on the arxive search page
retrieval_statuses = {model: [] for model in models}

for model in models.keys():
    params = {
        "model": models[model],
        "web_query_max_new_t": 50,
        "titles_limit": 5,
        "top_k_titles": 5,
        "chunk_size": 200,
        "chunk_overlap_fraction": 0.2,
        "top_k_chunks": 5,
        "answer_max_new_t": 500,
        "classifier_max_new_t": 10,
        "classifier": classifier
    }
    rag = RAGFlow(params)
    for question in questions:
        start = perf_counter()
        query = rag._generate_arxiv_query(question)
        latency = perf_counter() - start
        latencies[model].append(latency)
        embeddings = eval_model.encode(
            [question, query],
            normalize_embeddings=True
        )
        similarity = np.dot(
            embeddings[0],
            embeddings[1]
        )
        relevance_scores[model].append(float(similarity))
        top_titles = rag._retrieve_top_k_titles(query, question)
        if len(top_titles) > 0:
            retrieval_statuses[model].append(1)
        else:
            retrieval_statuses[model].append(0)

2026-10-09 19:20:08,032 | INFO | No device provided, using mps
2026-10-09 19:20:08,153 | INFO | HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
2026-10-09 19:20:08,169 | INFO | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/1110a243fdf4706b3f48f1d95db1a4f5529b4d41/modules.json?%2Fsentence-transformers%2Fall-MiniLM-L6-v2%2Fresolve%2Fmain%2Fmodules.json=&etag=%22952a9b81c0bfd99800fabf352f69c7ccd46c5e43%22 "HTTP/1.1 200 OK"
2026-10-09 19:20:08,255 | INFO | HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
2026-10-09 19:20:08,285 | INFO | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/1110a243fdf4706b3f48f1d95db1a4f5529b4d41/config_sentence_transformers.json?%2Fsentence-tr

In [21]:
evaluation_df = pd.DataFrame({
    model: {
        "query_relevance": np.mean(relevance_scores[model]),
        "query_latency": np.mean(latencies[model]),
        "retrieval_success_rate": np.mean(retrieval_statuses[model]),
    }
    for model in models
})

evaluation_df

,qwen,gpt
query_relevance,0.128255,1.000000
query_latency,6.236574,13.226605
retrieval_success_rate,1.000000,1.000000


The result above shows that gpt model takes twice longer to generate a query, however, its queries have  better cosine similarity to the original question than qwen's output. At this point, I do not have enough information to decide how important it is to have a query that would be perfectly similar to the user's question. I will need to do further exploration to make a decision on the model.
```
	qwen	gpt
query_relevance	0.128255	1.000000
query_latency	6.236574	13.226605
retrieval_success_rate	1.000000	1.000000
```